In [232]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [233]:
seed = 42

# Data

In [234]:
def prep_df(df: pd.DataFrame):
    df = df.drop(["ID", "is_fit"], axis=1, errors='ignore')

    # fix missing
    df['sleep_hours'] = df['sleep_hours'].fillna(df['sleep_hours'].mean())

    # dummy encoding
    df["gender"] = (df["gender"] == 'M').astype(np.int32)
    df["smokes"] = (df["smokes"] == 'yes').astype(np.int32)

    # feature engineering
    height_m = df["height_cm"] // 100
    df["BMI"] = df["weight_kg"] / (height_m ^ 2)

    return df

In [235]:
df = pd.read_csv("train_data.csv")
train_df = prep_df(df)

In [236]:
train_df.head()

,age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,BMI
0,35,177,56,63.4,113.1,8.5,0.07,1.68,0,1,18.666667
1,34,187,71,75.5,114.0,6.6,1.59,1.14,0,0,23.666667
2,67,158,101,76.4,113.2,8.1,1.36,1.16,1,0,33.666667
3,76,198,61,45.0,124.2,7.6,1.99,2.17,0,0,20.333333
4,45,156,111,69.7,116.2,8.8,8.44,1.94,1,1,37.000000


# EDA

In [237]:
df["sleep_hours"].isna().sum()

np.int64(121)

In [238]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600 entries, 0 to 1599
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1600 non-null   int64  
 1   height_cm          1600 non-null   int64  
 2   weight_kg          1600 non-null   int64  
 3   heart_rate         1600 non-null   float64
 4   blood_pressure     1600 non-null   float64
 5   sleep_hours        1600 non-null   float64
 6   nutrition_quality  1600 non-null   float64
 7   activity_index     1600 non-null   float64
 8   smokes             1600 non-null   int32  
 9   gender             1600 non-null   int32  
 10  BMI                1600 non-null   float64
dtypes: float64(6), int32(2), int64(3)
memory usage: 125.1 KB


# Model selection

In [239]:
X_train, X_test, y_train, y_test = train_test_split(train_df, df["is_fit"], test_size=0.33, random_state=seed)

In [240]:
def evaluate(clf):
    scores = cross_val_score(clf, X_train, y_train, scoring='f1', cv=3, n_jobs=-1)

    clf.fit(X_train, y_train)
    score = f1_score(y_test, clf.predict(X_test))
    
    return (np.mean(scores) - np.std(scores)).item(), score

In [250]:
lr = LogisticRegression(max_iter=1000)

evaluate(lr)

(0.6614173969964463, 0.7153652392947103)

In [251]:
svc = Pipeline([
    ('scale', RobustScaler()),
    ('clf',  SVC())
])

evaluate(svc)

(0.6573017000706798, 0.7093596059113301)

In [252]:
clf = lr

clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

# Submission

In [253]:
test_df_ = pd.read_csv("test_data.csv")
test_df = prep_df(test_df_)

In [254]:
# subtask 1
subtask1 = test_df["BMI"] > 25

# subtask 2
subtask2 = clf.predict(test_df)

In [255]:
def build_subtask(sid, answers):
    return pd.DataFrame({
        "subtaskID": sid,
        "datapointID": test_df_["ID"],
        "answer": answers
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_subtask(sid, ans) for (sid, ans) in subtasks])

In [256]:
submission.head()

,subtaskID,datapointID,answer
0,1,1861,0
1,1,354,0
2,1,1334,0
3,1,906,1
4,1,1290,1


In [257]:
submission.to_csv("submission.csv", index=False)